# Augment nominal images, then train the predictor

This separate workflow applies deterministic image-space augmentation only to the nominal training portion. Original validation images and the official test split remain untouched. Augmented images, latent caches, and checkpoints use separate roots from the baseline workflow.

In [ ]:
from pathlib import Path
import random
import sys
import torch
import yaml
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
cfg = yaml.safe_load((PROJECT_ROOT / 'configs' / 'augmentation' / 'base.yaml').read_text())
data_cfg = yaml.safe_load((PROJECT_ROOT / 'configs' / 'data' / 'mvtec.yaml').read_text())
AUGMENTATION_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'augmentation' / 'base.yaml'
predictor_cfg = yaml.safe_load((PROJECT_ROOT / 'configs' / 'predictor' / 'base.yaml').read_text())
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'mvtec_ad'
if not DATASET_ROOT.is_dir(): DATASET_ROOT = PROJECT_ROOT / 'dataset'
IMAGE_ROOT = PROJECT_ROOT / cfg['output']['image_root']
CACHE_ROOT = PROJECT_ROOT / cfg['output']['cache_root']
CHECKPOINT_ROOT = PROJECT_ROOT / cfg['output']['checkpoint_root']
CATEGORIES = ['bottle', 'capsule', 'hazelnut']
# The nominal split is independent of augmentation settings (configs/data/mvtec.yaml).
VAL_FRACTION = float(data_cfg['nominal_split']['val_fraction'])
SPLIT_SEED = int(data_cfg['nominal_split']['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_MOCK_ENCODER = False
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'IN22K-vit.g.16-600e.pth.tar'
OFFICIAL_REPO_PATH = PROJECT_ROOT / 'external' / 'ijepa'
print(f'Writing augmented images to {IMAGE_ROOT}\nDevice: {DEVICE}')
if DEVICE == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from ijepa_ad.data.augmentation import load_augmentation_config, materialize_augmented_records
from ijepa_ad.data.mvtec_dataset import MVTecADDataset
from ijepa_ad.data.splits import split_nominal_train_val
augmented_records, aug_configs, original_train_counts = {}, {}, {}
for category in CATEGORIES:
    original = MVTecADDataset(DATASET_ROOT, category, 'train').samples
    split = split_nominal_train_val(original, val_fraction=VAL_FRACTION, seed=SPLIT_SEED)
    aug_cfg = load_augmentation_config(AUGMENTATION_CONFIG_PATH, category)
    # Images live in a directory named by the settings fingerprint; ids are content hashes,
    # so a changed setting yields new files and new cache sample ids (no stale reuse).
    records = materialize_augmented_records(split.train, aug_cfg, category=category, output_root=IMAGE_ROOT)
    augmented_records[category] = (records, split.val)
    aug_configs[category], original_train_counts[category] = aug_cfg, len(split.train)
    print(f'{category}: original train={len(split.train)}, augmented train={len(records)}, val={len(split.val)}, fingerprint={aug_cfg.fingerprint()}, hflip_p={aug_cfg.horizontal_flip_probability}')


## Encode augmented training data and original validation data

In [ ]:
from ijepa_ad.caching.builder import build_latent_cache
from ijepa_ad.caching.cli import MockLatentEncoder, mock_checkpoint_sha256
from ijepa_ad.caching.latent_cache import compute_file_sha256
from ijepa_ad.data.transforms import PreprocessConfig
from ijepa_ad.encoder.ijepa_wrapper import IJEPAEncoder
if USE_MOCK_ENCODER:
    encoder, encoder_name, checkpoint_sha256, embedding_dim = MockLatentEncoder(predictor_cfg['architecture']['embedding_dim'], SPLIT_SEED), 'mock_encoder', mock_checkpoint_sha256(SPLIT_SEED), predictor_cfg['architecture']['embedding_dim']
else:
    encoder, encoder_name, checkpoint_sha256, embedding_dim = IJEPAEncoder(CHECKPOINT_PATH, official_repo_path=OFFICIAL_REPO_PATH, device=DEVICE), 'ijepa_vit_g16_in22k_ep44', compute_file_sha256(CHECKPOINT_PATH), predictor_cfg['architecture']['embedding_dim']
class DeviceEncoder:
    def encode(self, batch): return encoder.encode(batch.to(DEVICE)).cpu()
device_encoder = DeviceEncoder()
for category, (train_records, val_records) in augmented_records.items():
    for split, records in [('train', train_records), ('val', val_records)]:
        build_latent_cache(records, device_encoder, cache_root=CACHE_ROOT, category=category, split=split, encoder_name=encoder_name, checkpoint_sha256=checkpoint_sha256, patch_size=16, embedding_dim=embedding_dim, config=PreprocessConfig(), batch_size=8)
    print(f'{category}: augmented latent cache ready')
del device_encoder, encoder
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Train and save augmented-data checkpoints

In [ ]:
import time
from dataclasses import asdict
from ijepa_ad.caching.cache_schema import CacheExpectations, PreprocessingSpec
from ijepa_ad.predictor.cache_dataset import load_cached_latent_dataset
from ijepa_ad.predictor.transformer_predictor import LatentTransformerPredictor, PredictorConfig
from ijepa_ad.predictor.training import TrainingConfig, train_predictor
from ijepa_ad.data.transforms import PreprocessConfig
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
architecture, training = predictor_cfg['architecture'], predictor_cfg['training']
preprocessing = PreprocessingSpec.from_config(PreprocessConfig())
for category in CATEGORIES:
    def expected(split): return CacheExpectations(category=category, split=split, encoder_name=encoder_name, checkpoint_sha256=checkpoint_sha256, patch_size=16, embedding_dim=embedding_dim, preprocessing=preprocessing)
    train_ds, val_ds = load_cached_latent_dataset(CACHE_ROOT, expected('train')), load_cached_latent_dataset(CACHE_ROOT, expected('val'))
    model_cfg = PredictorConfig(embedding_dim=embedding_dim, num_tokens=train_ds.cache.metadata.num_tokens, hidden_dim=architecture['hidden_dim'], depth=architecture['depth'], num_heads=architecture['num_heads'], mlp_ratio=architecture['mlp_ratio'], dropout=architecture['dropout'])
    path = CHECKPOINT_ROOT / f'{category}_{training["loss_type"]}.pt'
    started = time.perf_counter()
    result = train_predictor(LatentTransformerPredictor(model_cfg), train_ds, val_ds, TrainingConfig(**{**training, 'device': DEVICE}), checkpoint_path=path)
    seconds = time.perf_counter() - started
    payload = torch.load(path, map_location='cpu', weights_only=True)
    assert payload['cache_provenance']['train']['split'] == 'train' and payload['cache_provenance']['val']['split'] == 'val'
    history, best_epoch, provenance = payload['history'], payload['best_epoch'], payload['cache_provenance']
    best, aug_cfg = history[best_epoch - 1], aug_configs[category]
    assert best['epoch'] == best_epoch
    n_train, n_original = len(provenance['train']['sample_ids']), original_train_counts[category]
    print(f'=== {category} ===')
    print(f'  checkpoint         : {path}  ({path.stat().st_size / 2**20:.2f} MiB)')
    print(f'  augmentation       : enabled   fingerprint: {aug_cfg.fingerprint()}')
    print(f'  augmentation config: {asdict(aug_cfg)}')
    print(f'  mask strategy      : {payload["training_config"]["mask_strategy"]}   loss: {payload["training_config"]["loss_type"]}')
    print(f'  training samples   : {n_train} = {n_original} original + {n_train - n_original} augmented copies   validation samples (original, unaugmented): {len(provenance["val"]["sample_ids"])}')
    print(f'  epochs run         : {len(history)} (max {payload["training_config"]["max_epochs"]})   best epoch: {best_epoch}')
    print(f'  train loss @ best  : {best["train_loss"]:.6f}   last epoch: {history[-1]["train_loss"]:.6f}')
    print(f'  val loss @ best    : {payload["best_val_loss"]:.6f}')
    print(f'  training duration  : {seconds:.1f} s (train_predictor call incl. checkpoint write)')